<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/accreted_title_step_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The goal of the project is to take in TCS backend data then for each title -  of

1.   get the min, mid, max
2.   count the number of steps in the scale

This project is to support the step calculation of accreted dtitles

In [4]:
import pandas as pd

tcs_backend_folder = "/content/drive/MyDrive/Data/UCOP_Job_Codes_Summary"

In [5]:
tcs_backend_file = pd.ExcelFile(f'{tcs_backend_folder}/R-349 Job Codes Summary_07282026.xlsx')
tcs_backend_file.sheet_names
#

['Job Code Review',
 'Represented Job Codes',
 'Non-Represented Job Codes',
 'Shift and On Call Rates']

In [6]:
tcs_backend_tabs = tcs_backend_file.sheet_names
represented_job_codes_df = tcs_backend_file.parse('Represented Job Codes', skiprows=9, skipfooter=2)
represented_job_codes_df.shape

(128803, 30)

In [7]:
represented_active_df = represented_job_codes_df[represented_job_codes_df['Salary Grade Eff Status'] == 'A']
represented_active_df.shape

(109454, 30)

In [8]:
upte_represented_job_codes_df = represented_active_df[represented_active_df['Union Code'].isin(['HX', 'RX', 'TX'])]
upte_represented_job_codes_df.shape

(40459, 30)

In [9]:
upte_job_title_Eff_date_cnt = upte_represented_job_codes_df.groupby(['Salary Plan SETID','Job Code Description'])['Eff Date - Salary Grade'].nunique().reset_index(name='Date Count')
multi_eff_date_setid_jobCode_cnt = upte_job_title_Eff_date_cnt.where(upte_job_title_Eff_date_cnt['Date Count'] > 1).dropna()

In [10]:
# get the latest effective date for each job title
latest_eff_date_df = upte_represented_job_codes_df.groupby(['Salary Plan SETID','Job Code Description'])['Eff Date - Salary Grade'].max().reset_index(name='Latest Effective Date')

In [12]:
# filtered UPTE represented job codes
titles_not_updated = upte_represented_job_codes_df[upte_represented_job_codes_df['Eff Date - Salary Grade'] < '2026-07-01']
titles_not_updated.shape   # 1192 rows


(1192, 30)

In [13]:
# Merge to get the latest effective date for each job title for which the effective date is before July 1, 2026
# WILL KEEP THIS ONE TO LATER TAKE CARE OF
titles_not_updated_w_max_date = titles_not_updated.merge(latest_eff_date_df, on=['Salary Plan SETID', 'Job Code Description'], how='inner')

# Drop the LBNL business units as they are not supposed to receive the 5% ATB in July 2026
titles_not_updated_w_max_date = titles_not_updated_w_max_date[titles_not_updated_w_max_date['Salary Plan SETID'] != 'LBNL1']
titles_not_updated_w_max_date.shape       # 1130 rows

(1130, 31)

In [14]:

# Filter the rows with the latest effective date for each job title that have multiple effective dates
# THIS DF WILL BE CONCATENATED LATER TO CREATE THE FINAL DF WITH THE LATEST EFFECTIVE DATE

upte_represented_max_eff_date_df = upte_represented_job_codes_df\
    .merge(multi_eff_date_setid_jobCode_cnt,
           on=['Salary Plan SETID', 'Job Code Description'],
           how='inner')\
    .sort_values('Eff Date - Salary Grade', ascending=False)\
    .drop_duplicates(['Salary Plan SETID', 'Job Code Description', 'UCPATH Step', 'UC  Half Step'],
                     keep='first')

upte_represented_max_eff_date_df.shape

(1566, 31)

Codes above are all from the [TCS payscale notebook](https://colab.research.google.com/drive/1MwFYF32tmy4eGs_cS6p0rc_hfDQ3drTM?usp=chrome_ntp#scrollTo=H9473xBkUotq)